# AI Financial Life Coach (AFILC) — Cuaderno de Modelado Reproducible
### Asignatura 7 · Trabajo Fin de Máster · Grupo 05
### Master en Big Data & Business Intelligence — Next Educación / Universidad Isabel I de Burgos

---

Este cuaderno documenta y reproduce la fase de **modelado analítico** del proyecto AFILC
sobre el dataset sintético de transacciones Tink (`dataset_final_usuarios_tink.csv`,
~640.000 registros mensuales de 40.000 usuarios).

## Mapeo con la rúbrica de la asignatura

| Apartado rúbrica | Peso | Sección de este cuaderno |
|---|---|---|
| 4.1 Preprocesamiento de datos | 20% | [§1](#1.-Preprocesamiento-de-datos-(20%)) |
| 4.2 Modelos Predictivos Principales | 70% | [§2](#2.-Modelos-Predictivos-Principales-(70%)) |
| 4.3 Modelo Avanzado o Extensión | 5% | [§3](#3.-Modelo-Avanzado-o-Extensión-(5%)---Modelo-temporal) |
| 4.4 Análisis No Supervisado o Segmentación | 5% | [§4](#4.-Análisis-No-Supervisado---Segmentación-de-usuarios-(5%)) |

## Procedencia de los datos y reproducibilidad

El dataset de entrada (`data/clean/dataset_final_usuarios_tink.csv`) es la salida final de
un pipeline previo de generación y preparación de datos sintéticos Tink
(`01_pipeline_tink/06_prepara_usuarios_tink.py`, y los scripts numerados anteriores que
generan y enriquecen las transacciones sintéticas). Ese pipeline completo **no se
reejecuta dentro de este cuaderno** por dos motivos:

1. Parte de archivos intermedios de varios GB (transacciones sintéticas crudas) que no
   se distribuyen en el repositorio por su tamaño.
2. Su tiempo de ejecución completo es de aproximadamente 10-15 minutos, frente a los
   pocos minutos que requiere la fase de modelado aquí documentada.

Este cuaderno toma como punto de partida el dataset ya preparado (panel usuario-mes con
`perfil`, `salario`, `gasto_total`, `ahorro`, categorías de gasto, IPC, etc.) y es
**completamente reproducible a partir de él**: cada celda de código ejecuta directamente
los scripts de producción del proyecto (`04_modelos_predictivos/*.py`), que son los
mismos que generan los artefactos (gráficos, métricas) usados en la memoria. No hay
lógica duplicada entre este cuaderno y el código de producción.

**Requisitos:** Python 3.10+, `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`,
`scipy`.


In [ ]:
import os
import sys
import json
import subprocess
import pandas as pd
from IPython.display import Image, display, Markdown

# Raíz del repositorio (ajustar si el cuaderno se ejecuta desde otra ubicación)
ROOT = os.getcwd()
DATA_CLEAN = os.path.join(ROOT, "data", "clean")
MODELOS_DIR = os.path.join(ROOT, "04_modelos_predictivos")
PIPELINE_DIR = os.path.join(ROOT, "01_pipeline_tink")

assert os.path.exists(os.path.join(DATA_CLEAN, "dataset_final_usuarios_tink.csv")), (
    "No se encuentra dataset_final_usuarios_tink.csv en data/clean/. "
    "Este cuaderno espera ejecutarse desde la raíz del repositorio, con el dataset "
    "ya generado por 01_pipeline_tink/06_prepara_usuarios_tink.py."
)

def ejecutar_script(nombre_script, carpeta=MODELOS_DIR, timeout=1800):
    """Ejecuta un script de producción del proyecto como subproceso (idéntico a como
    se ejecuta desde la línea de comandos) y muestra su salida estándar. Lanza un error
    si el script termina con código distinto de cero, para que un fallo en el pipeline
    de modelado no pase desapercibido."""
    ruta = os.path.join(carpeta, nombre_script)
    print(f">>> Ejecutando {ruta}\n")
    resultado = subprocess.run(
        [sys.executable, ruta], cwd=carpeta,
        capture_output=True, text=True, timeout=timeout,
    )
    print(resultado.stdout)
    if resultado.returncode != 0:
        print(resultado.stderr)
        raise RuntimeError(f"{nombre_script} terminó con código {resultado.returncode}")
    return resultado

def mostrar(nombre_png, ancho=850):
    display(Image(filename=os.path.join(DATA_CLEAN, nombre_png), width=ancho))


## 1. Preprocesamiento de datos (20%)

Las decisiones de preprocesamiento están aisladas en `04_modelos_predictivos/preprocesamiento.py`
y se reutilizan (no se duplican) desde `modelos_predictivos.py`, garantizando que el
modelado use exactamente el mismo preprocesamiento aquí documentado.

Se cubren tres decisiones, cada una justificada con un diagnóstico cuantitativo:

1. **Codificación de `perfil`** (variable categórica nominal: junior/medio/senior/freelance)
   mediante **one-hot encoding** (`drop_first=True`), no ordinal — `perfil` no tiene un
   orden natural entre sus categorías, por lo que una codificación ordinal induciría una
   relación de magnitud inexistente en los datos.
2. **Evaluación de asimetría (skewness)** de las variables monetarias y **transformación
   logarítmica (log1p)** donde está empíricamente justificada (regla: `|skew| > 1`).
3. **Escalado (StandardScaler)**, ajustado exclusivamente sobre el conjunto de
   entrenamiento, para evitar fuga de información del conjunto de test.


In [ ]:
ejecutar_script("preprocesamiento.py")


### Resultado: evaluación de asimetría

| Variable | Skew original | ¿Se transforma? |
|---|---|---|
| `salario` | 1.06 | Sí (log1p) |
| `gasto_total` | 13.01 | Sí (log1p) |
| `ahorro` | — (no evaluada) | No — contiene valores negativos y es la variable objetivo del Modelo 1, donde se prioriza la interpretabilidad de los coeficientes en euros |

`gasto_total` presentaba una asimetría extrema (13.01): unos pocos usuarios con gasto
mensual muy superior a la media distorsionaban la distribución. Tras `log1p`, su skew
se reduce a 1.61 — una mejora sustancial que acerca la variable a una distribución
manejable por un modelo lineal.


In [ ]:
mostrar("preprocesamiento_log_transform.png")


El panel compara, para cada variable, el histograma original (izquierda, rojo) con el
histograma tras `log1p` (derecha, azul). En ambos casos la transformación comprime la
cola derecha (usuarios con salario o gasto extremos) y produce una distribución más
simétrica y cercana a la normal — la condición que un modelo lineal (Modelo 1, Modelo 2)
aprovecha mejor que una variable con cola larga sin transformar.


## 2. Modelos Predictivos Principales (70%)

Esta sección ejecuta `modelos_predictivos.py`, que entrena de forma secuencial:

- **Modelo 1** (regresión, `RidgeCV`): predice el ahorro mensual (€) de un usuario a
  partir de su histórico reciente (salario, gasto, ahorro rezagado, perfil, IPC).
- **Modelo 1-bis**: explicación local (por usuario) de la predicción del Modelo 1,
  descomponiendo la predicción en la contribución de cada variable.
- **Modelo 2** (clasificación, `LogisticRegressionCV`): clasifica a un usuario como
  `ahorro_adecuado` o `ahorro_insuficiente` según si su tasa de ahorro supera la
  mediana histórica.
- **Modelo 3** (ver [§3](#3.-Modelo-Avanzado-o-Extensión-(5%)---Modelo-temporal)) y la
  **capa prescriptiva** (ver más abajo), que también se generan en esta misma ejecución.

La partición train/test se hace con `GroupShuffleSplit` por `user_id` (80/20): ningún
usuario aparece simultáneamente en train y test, evitando fuga de información entre
los distintos meses de un mismo usuario.

**Nota de ejecución:** esta celda entrena `LogisticRegressionCV` con validación cruzada
de 5 particiones sobre ~640.000 filas; puede tardar varios minutos.


In [ ]:
ejecutar_script("modelos_predictivos.py")


In [ ]:
with open(os.path.join(DATA_CLEAN, "metricas_modelos.json")) as f:
    metricas = json.load(f)

print(f"Modelo 1 (RidgeCV)  — MAE: {metricas['modelo1_mae']:.2f} €  |  R²: {metricas['modelo1_r2']:.4f}  |  alpha: {metricas['modelo1_alpha_ridge']}")
print(f"Modelo 2 (LogReg CV) — Accuracy: {metricas['modelo2_accuracy']:.4f}  |  Clases: {metricas['modelo2_clases']}")
print(f"Usuarios train: {metricas['usuarios_train']:,}  |  Usuarios test: {metricas['usuarios_test']:,}")


### 2.1 Modelo 1 — Regresión del ahorro mensual


In [ ]:
mostrar("modelo1_regresion_lineal.png")


El modelo explica el **80% de la varianza** del ahorro mensual (R² = 0.80) con un error
medio absoluto de **~609 €**, usando únicamente información disponible al cierre del
mes anterior (sin fuga de información del mes a predecir).


In [ ]:
mostrar("modelo1_coeficientes.png")


**Nota metodológica — colinealidad entre `salario_lag1_log` y `salario_roll3_log`:**
estas dos variables están altamente correlacionadas entre sí (r = 0.96), ya que derivan
de la misma variable base (`salario`) en dos ventanas temporales distintas. Con una
regresión lineal simple (OLS) esto produce coeficientes inestables y de signo
económicamente contraintuitivo en una de las dos variables, pese a que **ambas
correlacionan positivamente con el ahorro de forma individual** (r = +0.85 entre
`salario_roll3_log` y `ahorro`). Es un caso clásico de colinealidad, no un error en los
datos ni en el modelo.

Se opta por **`RidgeCV`** (regularización L2, con la fuerza `alpha` seleccionada por
validación cruzada) en lugar de `LinearRegression`: estabiliza las estimaciones de
coeficientes ante variables correlacionadas sin descartar ninguna variable y sin perder
la interpretabilidad lineal completa. La predicción agregada (MAE, R²) no se ve afectada
por este fenómeno — el efecto es únicamente sobre la estabilidad individual de estos dos
coeficientes concretos, y se documenta aquí de forma transparente en vez de ocultarse.


In [ ]:
print(metricas['modelo1_nota_colinealidad'])


### 2.2 Modelo 1-bis — Explicación local de una predicción


In [ ]:
mostrar("modelo1_explicacion_local_ejemplo.png")
mostrar("modelo1bis_explicacion_usuario_ejemplo.png")


Para un usuario concreto, se descompone la predicción del Modelo 1 en la contribución
(coeficiente × valor estandarizado) de cada variable — útil para explicar a un usuario
final *por qué* el modelo predice un ahorro determinado, un requisito de interpretabilidad
relevante en un asistente financiero dirigido a personas, no solo a analistas.


### 2.3 Modelo 2 — Clasificación del perfil de ahorro

**Historial de la decisión de clase (documentado también en la memoria):** el esquema
de clases de este modelo pasó por cuatro iteraciones antes de llegar al esquema binario
actual, cada una motivada por un diagnóstico cuantitativo real:

1. **Umbral absoluto fijo** (15%/5%) → colapsaba todos los usuarios en una única clase,
   porque la distribución real de Tink es atípica (mediana de tasa de ahorro ≈ 88%).
2. **Terciles (3 clases)** → resolvía el colapso, pero la clase central
   (`ahorro_moderado`) quedaba en una franja de solo ~4 puntos porcentuales —
   estadísticamente no separable (recall = 0.02, prácticamente inútil).
3. **3 clases de igual anchura** (8%/20%/72% de los usuarios) → mejoraba el recall de
   la clase central (0.65), pero hacía colapsar la precisión de `ahorro_insuficiente`
   (0.52 → 0.20, F1 0.61 → 0.32) — un nuevo problema, no una solución limpia.
4. **Esquema binario por mediana (adoptado)**: `ahorro_adecuado` / `ahorro_insuficiente`,
   con el umbral de decisión en la mediana histórica de la tasa de ahorro. Un único
   límite de decisión elimina el problema estructural de una clase atrapada entre dos
   fronteras, sin el trade-off de precisión observado en la alternativa de 3 clases.

**Resultado: F1 macro 0.72 (frente a 0.43 con terciles) — la mejora más significativa
de todo el proceso de modelado.**


In [ ]:
mostrar("modelo2_confusion.png")


La matriz de confusión es ahora claramente **diagonal-dominante** (la mayoría de los
aciertos están en la diagonal principal), a diferencia del esquema de terciles original,
donde la clase central quedaba casi completamente mal clasificada.


In [ ]:
mostrar("modelo2_coeficientes_por_clase.png")


Positivo = mayor probabilidad de `ahorro_adecuado`; negativo = mayor probabilidad de
`ahorro_insuficiente`. A diferencia del caso multiclase (*one-vs-rest*), con solo dos
clases `LogisticRegressionCV` devuelve un único vector de coeficientes (log-odds de la
clase positiva frente a la de referencia), por lo que el gráfico muestra un solo panel.


In [ ]:
print(metricas['modelo2_esquema_clases'])


### Capa prescriptiva

A partir de la clasificación del Modelo 2, se genera una recomendación de ajuste de
gasto por categoría para los usuarios clasificados como `ahorro_insuficiente`, con el
objetivo de acercar su tasa de ahorro a la mediana histórica (`umbral_mediana_pct`).


In [ ]:
mostrar("prescriptivo_categoria_sugerida.png")
print(f"\nUsuarios con recomendación de ajuste: {metricas['prescriptivo_usuarios_con_ajuste']:,} "
      f"({metricas['prescriptivo_pct_usuarios_con_ajuste']}% del total)")
print(f"Cobertura media de la brecha de ahorro tras el ajuste sugerido: "
      f"{metricas['prescriptivo_cobertura_media_brecha_pct']}%")


In [ ]:
recomendaciones = pd.read_csv(os.path.join(DATA_CLEAN, "recomendaciones_prescriptivas.csv"))
recomendaciones.head(10)


## 3. Modelo Avanzado o Extensión (5%) — Modelo temporal

El Modelo 3 proyecta el **ahorro medio agregado** de la base de usuarios a 6 meses vista,
satisfaciendo la opción *"modelo temporal"* del apartado 4.3 de la rúbrica.

**Nota metodológica:** la composición de usuarios activos varía significativamente de un
mes a otro en el dataset sintético de Tink (entre ~5.000 y ~30.000 usuarios según el mes,
por las distintas ventanas temporales de los 8 perfiles reales de origen). Una simple
media transversal mes a mes mezclaría este efecto de composición cambiante con la
evolución real del ahorro. Para corregirlo, se construye un **índice encadenado
(chain-linked index)**: en cada transición de mes a mes, se calcula la variación del
ahorro medio **solo sobre los usuarios presentes en ambos meses**, y esas variaciones se
encadenan multiplicativamente — la misma lógica que un índice de precios encadenado
(análoga a un IPC encadenado), de ahí el nombre.


In [ ]:
mostrar("modelo3_comparacion_naive_vs_encadenado.png")


El gráfico compara la media transversal simple (ingenua) con el índice encadenado
corregido: la versión ingenua muestra saltos artificiales causados por el cambio de
composición de la muestra, mientras que el índice encadenado aísla la tendencia real.


In [ ]:
mostrar("modelo3_proyeccion.png")

print("\nProyección del ahorro medio agregado a 6 meses:")
for p in metricas['modelo3_predicciones']:
    print(f"  {p['fecha']}: {p['ahorro_predicho']:,.2f} €")
print(f"\n{metricas['modelo3_metodo']}")


## 4. Análisis No Supervisado — Segmentación de usuarios (5%)

Se aplica **K-Means** sobre un perfil comportamental agregado **por usuario** (no por
usuario-mes), para obtener segmentos estables en el tiempo — rasgo de comportamiento del
usuario, no de un mes puntual.

**Variables de segmentación:** ingreso medio, tasa de ahorro media, volatilidad del
ahorro (coeficiente de variación relativo al propio ingreso), edad, composición del
gasto por categoría (% del gasto total en cada categoría) y perfil (one-hot).

El número de clústeres (**k=6**) se selecciona maximizando el coeficiente de silueta en
el rango k=3..6 (se excluye k=2 por separar trivialmente solo por nivel de ingreso, sin
aportar información nueva sobre el comportamiento de ahorro).


In [ ]:
ejecutar_script("segmentacion_usuarios.py")


In [ ]:
mostrar("segmentacion_evaluacion_k.png")


In [ ]:
with open(os.path.join(DATA_CLEAN, "segmentacion_diagnostico.json")) as f:
    seg = json.load(f)

print(f"k elegido: {seg['k_elegido']}  (silueta = {seg['silueta_k_elegido']:.4f})")
print(f"Criterio: {seg['criterio_seleccion_k']}")
print(f"Usuarios segmentados: {seg['n_usuarios_segmentados']:,}\n")

resumen = pd.DataFrame(seg['resumen_por_cluster']).set_index('cluster')
resumen[['pct_usuarios', 'ingreso_medio', 'tasa_ahorro_media', 'volatilidad_ahorro', 'ahorro_medio', 'n_usuarios']]


In [ ]:
mostrar("segmentacion_clusters_pca.png")


Proyección de los 6 clústeres sobre las 2 primeras componentes principales
(50.6% de la varianza explicada) — usada únicamente para visualización, no para el
propio clustering (que opera sobre el espacio completo de variables).


In [ ]:
mostrar("segmentacion_perfil_clusters.png")


### Interpretación de los segmentos

| Clúster | % usuarios | Ingreso medio | Tasa ahorro | Perfil dominante | Lectura |
|---|---|---|---|---|---|
| 0 | 24.8% | 7.544 € | 90.8% | Senior | Ahorradores de altos ingresos, muy estables |
| 1 | 12.4% | 2.096 € | 30.8% | Medio | Ingresos medios, capacidad de ahorro limitada |
| 2 | 45.0% | 2.006 € | 52.6% | Mixto | Grupo mayoritario, ahorro moderado |
| 3 | 12.4% | 2.006 € | 85.8% | Medio | Ahorradores disciplinados con ingresos bajos-medios |
| 4 | 5.1% | 1.622 € | **−153.2%** | Mixto | **Usuarios en riesgo financiero** (gasto sistemáticamente por encima del ingreso, alta volatilidad) |
| 5 | 0.3% | 3.566 € | 73.0% | Freelance | Nicho freelance, ingresos irregulares pero ahorro alto |

El **Clúster 4** (5.1% de los usuarios, tasa de ahorro media de −153%) identifica un
segmento de usuarios financieramente vulnerables — precisamente el tipo de usuario al
que un asistente financiero como AFILC debería priorizar en sus recomendaciones.

**Limitación de datos a documentar:** los Clústeres 2 y 4 muestran `pct_gasto_otros = 1.0`
(100% del gasto categorizado como "otros"). Esto es consistente con la limitación ya
documentada en `sgbd/docs/tink_mapping.md`: la categorización real de transacciones Tink
no se consultó para estos perfiles sintéticos, por lo que su gasto no está desglosado por
categoría. No afecta a la validez de la segmentación por ingreso/ahorro/volatilidad, pero
limita el detalle de la composición de gasto para estos dos segmentos específicos.


## Conclusiones

| Apartado rúbrica | Evidencia en este cuaderno |
|---|---|
| **4.1** Preprocesamiento (20%) | One-hot encoding de `perfil`, evaluación de asimetría + log-transform justificado empíricamente, escalado sin fuga de información (§1) |
| **4.2** Modelos predictivos (70%) | Modelo 1 (regresión, RidgeCV, R²=0.80) + Modelo 2 (clasificación binaria, LogisticRegressionCV, F1 macro=0.72) + capa prescriptiva (§2) |
| **4.3** Modelo avanzado (5%) | Modelo 3 — proyección temporal con índice encadenado (§3) |
| **4.4** No supervisado / segmentación (5%) | K-Means, k=6, silueta=0.49, 6 segmentos de comportamiento financiero interpretados (§4) |

Todas las decisiones de preprocesamiento y modelado están respaldadas por un
diagnóstico cuantitativo explícito (asimetría, correlación, silueta, F1 por esquema de
clases) en vez de aplicarse como reglas generales sin comprobar — y los casos donde un
modelo presenta una limitación conocida (colinealidad en Modelo 1, categorización de
gasto incompleta en 2 de 6 clústeres) se documentan de forma transparente en vez de
ocultarse, conforme al criterio de rigor metodológico de la rúbrica.
